# BONUS: GraphRAG toy con Cora/CiteSeer
## Master Oficial: Big Data Science

### Objetivos de aprendizaje
- Comparar GCN, GAT y GraphSAGE no solo por accuracy, sino por calidad de retrieval.
- Entender GraphRAG como retrieval de seeds + expansion estructural por vecinos.
- Interpretar visualmente por que cambia el contexto recuperado segun la arquitectura.

### Alcance de este notebook
- Incluye carga robusta de dataset (Plan A Cora, Plan B CiteSeer).
- Incluye entrenamiento rapido, retrieval, expansion, metricas y visualizacion.
- No depende de LLM para funcionar en clase.

**Tiempo estimado:** 20-30 minutos

### Checklist de salida
- [ ] Comparar test accuracy y train time en 3 arquitecturas.
- [ ] Medir hits@k, pureza tras expansion y latencia de consulta.
- [ ] Mostrar subgrafos recuperados y explicar diferencias.

## 0) Setup y reproducibilidad

No instalamos paquetes dentro del notebook.
Si falla PyG, revisar setup y kernel activo.

Notas operativas para clase:
- `FAST_CLASS_MODE=True` reduce epocas para mantener una demo fluida.
- `GNN_FORCE_CPU=1` permite forzar CPU si hay inestabilidad de runtime en GPU.

In [1]:
import os
import time
import random
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

import torch
import torch.nn.functional as F
from IPython.display import display
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, GATv2Conv, SAGEConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Runtime knobs for class stability
FORCE_CPU = os.getenv("GNN_FORCE_CPU", "0") == "1"
FAST_CLASS_MODE = True
MAX_EPOCHS = 80 if FAST_CLASS_MODE else 120
PATIENCE = 12 if FAST_CLASS_MODE else 20
K_TOP = 8
EXPANSION_HOPS = 1
MAX_EXPANDED_NODES = 250
MAX_PLOT_NODES = 60

if FORCE_CPU:
    device = torch.device("cpu")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Seed: {SEED} | Device: {device} | Data dir: {DATA_DIR.resolve()}")
print(
    f"FAST_CLASS_MODE={FAST_CLASS_MODE} | MAX_EPOCHS={MAX_EPOCHS} | PATIENCE={PATIENCE} | "
    f"K_TOP={K_TOP} | HOPS={EXPANSION_HOPS}"
)

/home/darian/projects/GNN_MaterialesCurso/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Seed: 42 | Device: cpu | Data dir: /home/darian/projects/GNN_MaterialesCurso/data
FAST_CLASS_MODE=True | MAX_EPOCHS=80 | PATIENCE=12 | K_TOP=8 | HOPS=1


## 1) Carga robusta del dataset (Plan A/Plan B)

In [2]:
def load_planetoid_with_fallback(root: Path):
    attempts = ["Cora", "CiteSeer"]
    last_error = None
    for name in attempts:
        try:
            ds = Planetoid(root=str(root / "planetoid"), name=name)
            print(f"Dataset activo: {name}")
            return ds, name
        except Exception as exc:
            last_error = exc
            print(f"Fallo al cargar {name}: {type(exc).__name__}: {exc}")
    raise RuntimeError(f"No se pudo cargar Cora ni CiteSeer: {last_error}")

dataset, dataset_name = load_planetoid_with_fallback(DATA_DIR)
data = dataset[0].to(device)

print(
    f"nodes={data.num_nodes} | edges={data.num_edges} | features={dataset.num_features} | classes={dataset.num_classes}"
)
print(
    f"train={int(data.train_mask.sum())} | val={int(data.val_mask.sum())} | test={int(data.test_mask.sum())}"
)

Dataset activo: Cora
nodes=2708 | edges=10556 | features=1433 | classes=7
train=140 | val=500 | test=1000


## 2) Modelos uniformes y entrenamiento rapido

Mantenemos el mismo presupuesto de entrenamiento para comparar arquitecturas de forma justa.
Salida esperada: tabla con `test_acc`, `best_val_acc` y `train_time_s`.

In [4]:
class GCNNet(torch.nn.Module):
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__()
        self.conv1 = GCNConv(dim_in, dim_h)
        self.conv2 = GCNConv(dim_h, dim_out)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=0.5, training=self.training)
        h = self.conv1(h, edge_index)
        h = torch.relu(h)
        emb = h
        h = F.dropout(h, p=0.5, training=self.training)
        logits = self.conv2(h, edge_index)
        return logits, emb


class GATNet(torch.nn.Module):
    def __init__(self, dim_in, dim_h, dim_out, heads=4):
        super().__init__()
        self.gat1 = GATv2Conv(dim_in, dim_h, heads=heads)
        self.gat2 = GATv2Conv(dim_h * heads, dim_out, heads=1)

    def forward(self, x, edge_index):
        h = F.dropout(x, p=0.6, training=self.training)
        h = self.gat1(h, edge_index)
        h = F.elu(h)
        emb = h
        h = F.dropout(h, p=0.6, training=self.training)
        logits = self.gat2(h, edge_index)
        return logits, emb


class GraphSAGENet(torch.nn.Module):
    def __init__(self, dim_in, dim_h, dim_out):
        super().__init__()
        self.sage1 = SAGEConv(dim_in, dim_h)
        self.sage2 = SAGEConv(dim_h, dim_out)

    def forward(self, x, edge_index):
        h = self.sage1(x, edge_index)
        h = torch.relu(h)
        emb = h
        h = F.dropout(h, p=0.5, training=self.training)
        logits = self.sage2(h, edge_index)
        return logits, emb


@dataclass
class TrainResult:
    name: str
    model: torch.nn.Module
    test_acc: float
    best_val_acc: float
    train_time_s: float
    logits: torch.Tensor
    embeddings: torch.Tensor


def accuracy(pred_y: torch.Tensor, y: torch.Tensor) -> float:
    return float((pred_y == y).sum().item() / len(y))


def train_with_early_stopping(model, data, lr=0.01, weight_decay=5e-4, max_epochs=120, patience=20):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = torch.nn.CrossEntropyLoss()

    best_state = None
    best_val_loss = float("inf")
    best_val_acc = 0.0
    wait = 0

    t0 = time.perf_counter()
    for _ in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        logits, _ = model(data.x, data.edge_index)
        train_loss = criterion(logits[data.train_mask], data.y[data.train_mask])
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits, _ = model(data.x, data.edge_index)
            val_loss = criterion(val_logits[data.val_mask], data.y[data.val_mask]).item()
            val_acc = accuracy(val_logits[data.val_mask].argmax(dim=1), data.y[data.val_mask])

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            wait = 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break

    train_time_s = time.perf_counter() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        logits, embeddings = model(data.x, data.edge_index)

    test_acc = accuracy(logits[data.test_mask].argmax(dim=1), data.y[data.test_mask])
    return model, test_acc, best_val_acc, train_time_s, logits.detach().cpu(), embeddings.detach().cpu()

In [5]:
configs = [
    ("GCN", GCNNet(dataset.num_features, 64, dataset.num_classes), 0.01),
    ("GAT", GATNet(dataset.num_features, 16, dataset.num_classes, heads=4), 0.01),
    ("GraphSAGE", GraphSAGENet(dataset.num_features, 64, dataset.num_classes), 0.01),
]

print(
    f"Training budget -> max_epochs={MAX_EPOCHS}, patience={PATIENCE}, "
    f"models={len(configs)}, dataset={dataset_name}"
)

results = []
for name, model, lr in configs:
    # Re-seed before each model for fair and reproducible comparison.
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model, test_acc, val_acc, train_time_s, logits, embeddings = train_with_early_stopping(
        model=model,
        data=data,
        lr=lr,
        weight_decay=5e-4,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
    )
    results.append(
        TrainResult(
            name=name,
            model=model,
            test_acc=test_acc,
            best_val_acc=val_acc,
            train_time_s=train_time_s,
            logits=logits,
            embeddings=embeddings,
        )
    )

metrics_df = pd.DataFrame([
    {
        "model": r.name,
        "test_acc": round(r.test_acc, 4),
        "best_val_acc": round(r.best_val_acc, 4),
        "train_time_s": round(r.train_time_s, 2),
    }
    for r in results
])
metrics_df.sort_values(by="test_acc", ascending=False).reset_index(drop=True)

Training budget -> max_epochs=80, patience=12, models=3, dataset=Cora


,model,test_acc,best_val_acc,train_time_s
0,GCN,0.799,0.790,2.46
1,GraphSAGE,0.788,0.758,2.99
2,GAT,0.784,0.794,3.12


## 3) Query evaluable: centroide de clase objetivo

Usamos una query en embedding space para que la comparacion sea estable y medible.
Interpretacion: "buscar nodos cercanos al concepto de la clase objetivo".

In [6]:
y_cpu = data.y.detach().cpu()
train_idx = torch.where(data.train_mask.detach().cpu())[0]
test_idx = torch.where(data.test_mask.detach().cpu())[0]

# Elegimos como clase objetivo la clase con mas nodos de entrenamiento.
classes, counts = torch.unique(y_cpu[train_idx], return_counts=True)
target_class = int(classes[counts.argmax()].item())

print(f"target_class={target_class}")
print(f"train_nodes_target={int((y_cpu[train_idx] == target_class).sum())}")
print(f"test_nodes_target={int((y_cpu[test_idx] == target_class).sum())}")

target_class=0
train_nodes_target=20
test_nodes_target=130


## 4) Retrieval top-k + expansion por grafo

In [7]:
edge_index_cpu = data.edge_index.detach().cpu()
neighbors = defaultdict(set)
for u, v in edge_index_cpu.t().tolist():
    neighbors[u].add(v)
    neighbors[v].add(u)


def retrieve_topk(embeddings: torch.Tensor, query_vec: torch.Tensor, candidate_index: torch.Tensor, k: int = 12):
    emb = embeddings[candidate_index]
    sims = F.cosine_similarity(emb, query_vec.unsqueeze(0), dim=1)
    k_eff = min(k, sims.numel())
    top_local = torch.topk(sims, k=k_eff).indices
    top_nodes = candidate_index[top_local]
    top_scores = sims[top_local]
    return top_nodes, top_scores


def expand_neighbors(seed_nodes, neighbors_map, hops=1, max_nodes=200):
    expanded = set(int(n) for n in seed_nodes)
    frontier = set(expanded)
    for _ in range(hops):
        nxt = set()
        for u in frontier:
            nxt.update(neighbors_map[u])
        expanded.update(nxt)
        frontier = nxt
        if len(expanded) >= max_nodes:
            break
    return sorted(expanded)


def cap_expanded_nodes(seeds, expanded_nodes, max_nodes):
    ordered = list(dict.fromkeys([*seeds, *expanded_nodes]))
    return ordered[:max_nodes]


def purity(nodes, labels, target):
    if len(nodes) == 0:
        return 0.0
    arr = labels[nodes]
    return float((arr == target).sum().item() / len(nodes))


retrieval_rows = []
retrieval_details = {}

for r in results:
    emb = r.embeddings
    query_vec = emb[train_idx[y_cpu[train_idx] == target_class]].mean(dim=0)

    t0 = time.perf_counter()
    seeds, seed_scores = retrieve_topk(emb, query_vec, test_idx, k=K_TOP)
    expanded = expand_neighbors(
        seeds.tolist(),
        neighbors,
        hops=EXPANSION_HOPS,
        max_nodes=MAX_EXPANDED_NODES,
    )
    expanded = cap_expanded_nodes(seeds.tolist(), expanded, MAX_EXPANDED_NODES)
    latency_ms = (time.perf_counter() - t0) * 1000.0

    hits_at_k = purity(seeds.tolist(), y_cpu, target_class)
    purity_expanded = purity(expanded, y_cpu, target_class)
    expansion_factor = len(expanded) / max(1, len(seeds))

    retrieval_rows.append({
        "model": r.name,
        "hits@k": round(hits_at_k, 4),
        "purity_expanded": round(purity_expanded, 4),
        "expansion_factor": round(expansion_factor, 2),
        "query_latency_ms": round(latency_ms, 2),
        "expanded_nodes": len(expanded),
    })

    retrieval_details[r.name] = {
        "seeds": [int(n) for n in seeds.tolist()],
        "seed_scores": [float(s) for s in seed_scores.tolist()],
        "expanded": expanded,
    }

retrieval_df = pd.DataFrame(retrieval_rows).sort_values(by="hits@k", ascending=False).reset_index(drop=True)
retrieval_df

,model,hits@k,purity_expanded,expansion_factor,query_latency_ms,expanded_nodes
0,GCN,1.0,0.9556,5.62,19.44,45
1,GAT,1.0,0.9787,5.88,0.27,47
2,GraphSAGE,1.0,0.9091,4.12,0.19,33


In [8]:
for model_name, detail in retrieval_details.items():
    seeds = detail["seeds"]
    scores = detail["seed_scores"]
    labels = [int(y_cpu[n]) for n in seeds]
    table = pd.DataFrame({
        "node_id": seeds,
        "score": [round(s, 4) for s in scores],
        "label": labels,
    })
    print(f"\nTop-k seeds ({model_name})")
    display(table.head(12))


Top-k seeds (GCN)


,node_id,score,label
0,2182,0.9913,0
1,2054,0.9891,0
2,2055,0.9890,0
3,2198,0.9890,0
4,2074,0.9877,0
5,2133,0.9874,0
6,2197,0.9855,0
7,2312,0.9838,0



Top-k seeds (GAT)


,node_id,score,label
0,2182,0.9808,0
1,1922,0.9732,0
2,2198,0.9699,0
3,2056,0.9692,0
4,1921,0.9675,0
5,2180,0.9673,0
6,2054,0.9658,0
7,2172,0.9631,0



Top-k seeds (GraphSAGE)


,node_id,score,label
0,2072,0.9676,0
1,2311,0.9496,0
2,1921,0.9495,0
3,2219,0.9483,0
4,1925,0.9452,0
5,1922,0.9428,0
6,2133,0.9412,0
7,2054,0.9406,0


## 5) Visualizacion del subgrafo expandido

In [ ]:
def plot_expanded_subgraph(model_name, detail, labels, neighbors_map, max_nodes=60):
    seeds = detail["seeds"]
    seed_scores = detail["seed_scores"]
    expanded = detail["expanded"]

    node_set = list(seeds)
    for n in expanded:
        if n not in node_set:
            node_set.append(n)
        if len(node_set) >= max_nodes:
            break

    G = nx.Graph()
    G.add_nodes_from(node_set)
    for u in node_set:
        for v in neighbors_map[u]:
            if v in G:
                G.add_edge(u, v)

    score_map = {n: 0.0 for n in node_set}
    for n, s in zip(seeds, seed_scores):
        score_map[n] = float(s)

    node_colors = [int(labels[n]) for n in G.nodes()]
    node_sizes = [220 + 1200 * max(0.0, score_map[n]) for n in G.nodes()]

    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(G, seed=42)
    nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.8)
    nodes = nx.draw_networkx_nodes(
        G,
        pos,
        node_color=node_colors,
        node_size=node_sizes,
        cmap=plt.cm.tab10,
        alpha=0.9,
        edgecolors="black",
        linewidths=0.3,
    )
    plt.colorbar(nodes, label="class label")
    plt.title(f"{model_name} | expanded subgraph (target_class={target_class})")
    plt.axis("off")
    plt.show()


for model_name in [r.name for r in results]:
    plot_expanded_subgraph(
        model_name,
        retrieval_details[model_name],
        y_cpu,
        neighbors,
        max_nodes=MAX_PLOT_NODES,
    )

## 6) Cierre: resumen estructurado (sin LLM por defecto)

GraphRAG en este bonus se resume en tres pasos:
- Seed retrieval: recuperar top-k nodos cercanos a la query en embedding space.
- Expansion por grafo: ampliar contexto con vecinos (hops sobre `edge_index`).
- Evaluacion: medir `hits@k`, `purity_expanded` y `expansion_factor` para cuantificar calidad y coste de contexto.

In [9]:
summary_df = metrics_df.merge(retrieval_df, on="model", how="left")
display(summary_df.sort_values(by="hits@k", ascending=False).reset_index(drop=True))

best_row = summary_df.sort_values(by=["hits@k", "purity_expanded", "test_acc"], ascending=False).iloc[0]

print("\nResumen interpretativo")
print(f"- Dataset activo: {dataset_name}")
print(f"- Clase objetivo: {target_class}")
print(f"- Mejor retrieval segun hits@k: {best_row['model']}")
print(
    f"- ({best_row['model']}) hits@k={best_row['hits@k']:.3f}, "
    f"purity_expanded={best_row['purity_expanded']:.3f}, "
    f"latency={best_row['query_latency_ms']:.2f} ms"
)
print("- Mensaje docente: la arquitectura cambia embeddings, y eso cambia retrieval y contexto expandido.")

,model,test_acc,best_val_acc,train_time_s,hits@k,purity_expanded,expansion_factor,query_latency_ms,expanded_nodes
0,GCN,0.799,0.790,2.46,1.0,0.9556,5.62,19.44,45
1,GAT,0.784,0.794,3.12,1.0,0.9787,5.88,0.27,47
2,GraphSAGE,0.788,0.758,2.99,1.0,0.9091,4.12,0.19,33



Resumen interpretativo
- Dataset activo: Cora
- Clase objetivo: 0
- Mejor retrieval segun hits@k: GAT
- (GAT) hits@k=1.000, purity_expanded=0.979, latency=0.27 ms
- Mensaje docente: la arquitectura cambia embeddings, y eso cambia retrieval y contexto expandido.


## 7) Modo opcional con LLM (desactivado por defecto)

- Si `USE_LLM=True` y hay API key, puedes redactar un resumen natural.
- Si no, la salida estructurada anterior es la ruta oficial de clase.

In [ ]:
USE_LLM = False

if USE_LLM:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("OPENAI_API_KEY no encontrada. Se mantiene modo determinista.")
    else:
        print("LLM opcional habilitada: implementar prompt con contexto compacto si se requiere.")
else:
    print("USE_LLM=False -> ruta offline determinista activa.")